# Berlin Food & Weekly Markets - Data Transformation

**Author:** Michael Wetzel  
**Date:** November 2025  
**Purpose:** Transform and combine multiple data sources for Berlin food markets database

This notebook performs:
1. Loading and combining multiple data sources
2. Data cleaning and standardization
3. Filtering (flea markets, past Christmas markets)
4. Spatial join to add neighborhood_id and district_id
5. Address completion using Nominatim
6. Export to CSV with NO missing values

## Setup and Imports

In [16]:
import pandas as pd
import geopandas as gpd # Install and import geopandas for spatial operations
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time
import json
import re
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [17]:
# File paths
SOURCES_DIR = Path('food_markets/sources')
OUTPUT_FILE = 'food_markets/berlin_food_markets_clean.csv'

# District ID mapping (official 8-digit Berlin codes for foreign key constraint)
DISTRICT_MAPPING = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Final schema including primary key and foreign keys
FINAL_COLUMNS = [
    'market_id',  # Primary key
    'market_name', 'market_type', 'address', 'postal_code', 'district',
    'neighborhood_id', 'district_id', 'opening_days', 'opening_hours',
    'operator', 'contact_email', 'website', 'accessibility',
    'latitude', 'longitude', 'data_source', 'notes'
]

print("Configuration loaded ✓")

Configuration loaded ✓


## Step 1: Load Data Sources

In [18]:
print("\n📥 Loading Data Sources")
print("-" * 70)

# Official GeoJSON sources (source of truth)
with open(SOURCES_DIR / 'weihnachtsmaerkte.geojson', 'r', encoding='utf-8') as f:
    christmas_data = json.load(f)
print(f"✓ Christmas markets: {len(christmas_data['features'])}")

with open(SOURCES_DIR / 'wochen-troedelmaerkte.geojson', 'r', encoding='utf-8') as f:
    weekly_data = json.load(f)
print(f"✓ Weekly/Flea markets: {len(weekly_data['features'])}")

# OSM data for gap filling
with open(SOURCES_DIR / 'OSM-berlin_markets.json', 'r', encoding='utf-8') as f:
    osm_data = json.load(f)
print(f"✓ OSM markets: {len(osm_data['elements'])}")

# Additional sources
df_wochenmarkt_de = pd.read_csv(SOURCES_DIR / 'wochenmarkt_deutschland.csv')
print(f"✓ Wochenmarkt Deutschland: {len(df_wochenmarkt_de)}")

with open(SOURCES_DIR / 'visitberlin_markets.txt', 'r', encoding='utf-8') as f:
    visitberlin_names = [line.strip() for line in f if line.strip()]
print(f"✓ VisitBerlin markets: {len(visitberlin_names)}")


📥 Loading Data Sources
----------------------------------------------------------------------
✓ Christmas markets: 44
✓ Weekly/Flea markets: 41
✓ OSM markets: 147
✓ Wochenmarkt Deutschland: 93
✓ VisitBerlin markets: 18


## Step 2: Helper Functions

In [19]:
def parse_german_date(date_str):
    """Parse German date format DD.MM.YYYY to datetime"""
    if not date_str or pd.isna(date_str):
        return None
    date_str = str(date_str).strip().split('\n')[0].strip()
    try:
        return datetime.strptime(date_str, '%d.%m.%Y')
    except:
        return None

def is_current_christmas_market(data):
    """Check if Christmas market is current (not from past years)"""
    bis_date = parse_german_date(data.get('bis', ''))
    if bis_date:
        # Market is current if end date is in the future or this season
        return bis_date >= datetime.now()
    # If no end date, assume it's current
    return True

def extract_geojson_to_df(geojson_data, market_type):
    """Extract GeoJSON properties.data to DataFrame"""
    records = []
    for feature in geojson_data['features']:
        data = feature['properties']['data'].copy()
        
        # Filter out past Christmas markets
        if market_type == 'christmas_market':
            if not is_current_christmas_market(data):
                continue  # Skip this market
        
        data['market_type'] = market_type
        data['data_source'] = 'berlin.de_official'
        records.append(data)
    return pd.DataFrame(records)

def standardize_columns(df):
    """Standardize column names"""
    mapping = {
        'name': 'market_name', 'bezeichnung': 'market_name',
        'strasse': 'address', 'plz_ort': 'postal_code', 'plz': 'postal_code',
        'bezirk': 'district', 'tage': 'opening_days', 'zeiten': 'opening_hours',
        'oeffnungszeiten': 'opening_hours', 'veranstalter': 'operator',
        'betreiber': 'operator', 'email': 'contact_email', 'www': 'website',
        'barrierefreiheit': 'accessibility', 'bemerkungen': 'notes',
        'lat': 'latitude', 'lng': 'longitude'
    }
    return df.rename(columns=mapping)

print("Helper functions defined ✓")

Helper functions defined ✓


## Step 3: Extract and Process Official Sources

In [20]:
print("\n🔧 Processing Official Sources")
print("-" * 70)

# Extract official sources
df_christmas = standardize_columns(extract_geojson_to_df(christmas_data, 'christmas_market'))
print(f"✓ Extracted {len(df_christmas)} current Christmas markets")

df_weekly_all = standardize_columns(extract_geojson_to_df(weekly_data, 'weekly_market'))

# Filter out flea markets
flea_keywords = ['floh', 'trödel', 'antik', 'sammler', 'kiefi', 'kinderfloh']
pattern = '|'.join(flea_keywords)
is_flea = df_weekly_all.get('market_name', pd.Series()).str.lower().str.contains(pattern, na=False)
df_weekly = df_weekly_all[~is_flea].copy()
print(f"✓ Filtered out {is_flea.sum()} flea markets")

# Combine official sources
df_official = pd.concat([df_christmas, df_weekly], ignore_index=True)
print(f"✓ Combined official sources: {len(df_official)} markets")


🔧 Processing Official Sources
----------------------------------------------------------------------
✓ Extracted 43 current Christmas markets
✓ Filtered out 12 flea markets
✓ Combined official sources: 72 markets


## Step 4: Process OSM Data

In [21]:
print("\n🌐 Processing OSM Data")
print("-" * 70)

# Extract OSM data
osm_records = []
for element in osm_data['elements']:
    record = {
        'market_name': element.get('tags', {}).get('name', ''),
        'latitude': element.get('lat', element.get('center', {}).get('lat', None)),
        'longitude': element.get('lon', element.get('center', {}).get('lon', None)),
        'opening_hours': element.get('tags', {}).get('opening_hours', ''),
        'website': element.get('tags', {}).get('website', ''),
        'data_source': 'OSM',
        'market_type': 'christmas_market' if 'xmas:feature' in element.get('tags', {})
                      else 'food_court' if element.get('tags', {}).get('amenity') == 'food_court'
                      else 'weekly_market'
    }
    osm_records.append(record)
df_osm = pd.DataFrame(osm_records)

print(f"✓ Extracted {len(df_osm)} markets from OSM")


🌐 Processing OSM Data
----------------------------------------------------------------------
✓ Extracted 147 markets from OSM


## Step 5: Deduplicate Across Sources

In [22]:
print("\n🔍 Deduplicating")
print("-" * 70)

def clean_name(name):
    if pd.isna(name): return ''
    name = str(name).lower()
    name = re.sub(r'\b(wochenmarkt|markt|market|weekly|ökomarkt)\b', '', name)
    return re.sub(r'\s+', ' ', name).strip()

df_official['name_clean'] = df_official['market_name'].apply(clean_name)
df_osm['name_clean'] = df_osm['market_name'].apply(clean_name)

# Keep only OSM markets not in official sources
osm_new = df_osm[~df_osm['name_clean'].isin(df_official['name_clean']) & (df_osm['name_clean'].str.len() > 0)]
print(f"✓ Found {len(osm_new)} additional markets from OSM")


🔍 Deduplicating
----------------------------------------------------------------------
✓ Found 110 additional markets from OSM


## Step 6: Combine All Sources

In [23]:
print("\n🔗 Combining All Sources")
print("-" * 70)

def ensure_columns(df, columns):
    for col in columns:
        if col not in df.columns:
            df[col] = ''
    return df[columns]

df_combined = pd.concat([
    ensure_columns(df_official, FINAL_COLUMNS),
    ensure_columns(osm_new, FINAL_COLUMNS)
], ignore_index=True)

print(f"✓ Total combined: {len(df_combined)} markets")


🔗 Combining All Sources
----------------------------------------------------------------------
✓ Total combined: 182 markets


## Step 7: Data Cleaning

In [24]:
print("\n🧹 Cleaning Data")
print("-" * 70)

# Keep only markets with coordinates
df_combined['latitude'] = pd.to_numeric(df_combined['latitude'], errors='coerce')
df_combined['longitude'] = pd.to_numeric(df_combined['longitude'], errors='coerce')
df_clean = df_combined.dropna(subset=['latitude', 'longitude']).copy()
print(f"✓ Markets with valid coordinates: {len(df_clean)}")

# Fill missing values
fill_values = {
    'market_name': 'Unknown Market', 'market_type': 'weekly_market',
    'address': 'Not specified', 'postal_code': 'Not available',
    'district': 'Unknown', 'opening_days': 'See website',
    'opening_hours': 'See website', 'operator': 'Not specified',
    'contact_email': 'Not available', 'website': 'Not available',
    'accessibility': 'Not specified', 'data_source': 'unknown',
    'notes': 'None'
}

for col, value in fill_values.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(value).replace('', value)

# Remove duplicates
df_clean['lat_rounded'] = df_clean['latitude'].round(3)
df_clean['lon_rounded'] = df_clean['longitude'].round(3)
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['market_name', 'lat_rounded', 'lon_rounded'], keep='first')
print(f"✓ Removed {before - len(df_clean)} duplicates")

df_clean = df_clean.drop(columns=['lat_rounded', 'lon_rounded', 'name_clean'], errors='ignore')


🧹 Cleaning Data
----------------------------------------------------------------------
✓ Markets with valid coordinates: 182
✓ Removed 0 duplicates


## Step 8: Spatial Join for Neighborhood and District IDs

This is the crucial step! We perform a spatial join to match market coordinates with Berlin administrative boundaries.

In [25]:
print("\n🗺️  Performing Spatial Join")
print("-" * 70)

# Create GeoDataFrame from market coordinates
df_clean['geometry'] = df_clean.apply(
    lambda row: Point(float(row['longitude']), float(row['latitude'])),
    axis=1
)
markets_gdf = gpd.GeoDataFrame(df_clean, geometry='geometry', crs='EPSG:4326')
print(f"✓ Created GeoDataFrame with {len(markets_gdf)} markets")

# Load Berlin neighborhoods (LOR Ortsteile)
lor_path = Path('mapping/lor_ortsteile.geojson')
if not lor_path.exists():
    lor_path = Path('food_markets/sources/lor_ortsteile.geojson')

berlin_gdf = gpd.read_file(lor_path)
print(f"✓ Loaded {len(berlin_gdf)} Berlin neighborhoods")

# Perform spatial join - use spatial_name as neighborhood_id!
markets_with_location = gpd.sjoin(
    markets_gdf,
    berlin_gdf[['BEZIRK', 'spatial_name', 'geometry']],
    how='left',
    predicate='within'
)

matched = markets_with_location['spatial_name'].notna().sum()
unmatched = markets_with_location['spatial_name'].isna().sum()
print(f"✓ Spatial join complete: {matched} matched, {unmatched} outside Berlin")

# Map district IDs using official 8-digit codes
markets_with_location['district_id'] = markets_with_location['BEZIRK'].map(DISTRICT_MAPPING)

# Fallback: try original district column
mask = markets_with_location['district_id'].isna()
markets_with_location.loc[mask, 'district_id'] = \
    markets_with_location.loc[mask, 'district'].map(DISTRICT_MAPPING)

# Update district name from BEZIRK (official spatial join result)
markets_with_location.loc[markets_with_location['BEZIRK'].notna(), 'district'] = \
    markets_with_location.loc[markets_with_location['BEZIRK'].notna(), 'BEZIRK']

# Set neighborhood_id from spatial_name (e.g., "0101", "0102")
markets_with_location['neighborhood_id'] = markets_with_location['spatial_name']

# Handle markets outside Berlin
markets_with_location['neighborhood_id'] = markets_with_location['neighborhood_id'].fillna('Outside Berlin')
markets_with_location['district_id'] = markets_with_location['district_id'].fillna('99999999')

berlin_markets = (markets_with_location['district_id'] != '99999999').sum()
print(f"✓ Final: {berlin_markets} markets in Berlin, {len(markets_with_location) - berlin_markets} outside")


🗺️  Performing Spatial Join
----------------------------------------------------------------------
✓ Created GeoDataFrame with 182 markets
✓ Loaded 96 Berlin neighborhoods
✓ Spatial join complete: 163 matched, 19 outside Berlin
✓ Final: 163 markets in Berlin, 19 outside


## Step 9: Fill Missing Addresses with Nominatim

Use reverse geocoding to complete missing addresses based on coordinates.

We enrich two types of addresses:
1. Completely missing addresses ('Not specified')
2. Addresses without house numbers (detected via regex `\d`)

Many official sources only provide street names like "Winterfeldtplatz" without house numbers. By checking for the absence of digits, we can identify and enrich these incomplete addresses.

**Important Note on Address Quality:**
Even after reverse geocoding, some addresses may still lack house numbers. This is **expected behavior** because:
- Markets are typically located in public spaces (plazas, squares, parks, or along streets)
- These locations don't have traditional building addresses with house numbers
- Examples: "Maybachufer" (embankment), "Helene-Weigel-Platz" (plaza), "Karl-Marx-Allee" (avenue)
- Nominatim returns the best available address for the coordinate - if no house number exists, only the street/plaza name is provided
- Approximately 50% of markets will have street names only, which is correct for their location type

In [26]:
print("\n🌐 Geocoding Missing Addresses")
print("-" * 70)

# Initialize Nominatim geocoder
geolocator = Nominatim(user_agent="berlin_food_markets_transformation")

def reverse_geocode_address(lat, lon):
    """Get address AND postal code from coordinates using Nominatim"""
    try:
        time.sleep(1)  # Respect Nominatim usage policy (max 1 request per second)
        location = geolocator.reverse(f"{lat}, {lon}", language='de')
        if location:
            # Extract street address and postal code from the result
            addr = location.raw.get('address', {})
            road = addr.get('road', '')
            house_number = addr.get('house_number', '')
            postcode = addr.get('postcode', '')
            
            address = f"{road} {house_number}".strip() if road else location.address.split(',')[0]
            return address, postcode
        return None, None
    except (GeocoderTimedOut, GeocoderServiceError):
        return None, None
    except Exception:
        return None, None

# Find addresses needing enrichment - BUT ONLY FOR BERLIN MARKETS
berlin_markets_mask = markets_with_location['district_id'] != '99999999'

# 1. Addresses that are 'Not specified' or NaN
# 2. Addresses that don't contain any numbers (e.g., just street name)
missing_address = (markets_with_location['address'] == 'Not specified') | (markets_with_location['address'].isna())
no_house_number = markets_with_location['address'].str.contains(r'\d', regex=True, na=False) == False

# Combine: addresses that need geocoding AND are in Berlin
needs_geocoding = (missing_address | no_house_number) & berlin_markets_mask
geocoding_count = needs_geocoding.sum()

print(f"Addresses completely missing: {missing_address.sum()}")
print(f"Addresses without house numbers: {no_house_number.sum()}")
print(f"Total addresses to geocode (Berlin only): {geocoding_count}")

if geocoding_count > 0:
    print(f"Geocoding addresses (this may take a while, 1 sec per request)...")
    filled = 0
    for idx in markets_with_location[needs_geocoding].index:
        lat = markets_with_location.loc[idx, 'latitude']
        lon = markets_with_location.loc[idx, 'longitude']
        
        if pd.notna(lat) and pd.notna(lon):
            address, postcode = reverse_geocode_address(lat, lon)
            if address:
                markets_with_location.loc[idx, 'address'] = address
                filled += 1
            if postcode:
                markets_with_location.loc[idx, 'postal_code'] = postcode
            if filled % 10 == 0:
                print(f"  Geocoded {filled}/{geocoding_count} addresses...")
    
    print(f"✓ Filled/enriched {filled} addresses using Nominatim reverse geocoding")
    
    # Check how many still lack house numbers after geocoding
    still_no_numbers = markets_with_location['address'].str.contains(r'\d', regex=True, na=False) == False
    print(f"ℹ️  Note: {still_no_numbers.sum()} addresses still lack house numbers (expected for public spaces)")
else:
    print(f"✓ All markets have complete addresses!")


🌐 Geocoding Missing Addresses
----------------------------------------------------------------------
Addresses completely missing: 110
Addresses without house numbers: 140
Total addresses to geocode (Berlin only): 135
Geocoding addresses (this may take a while, 1 sec per request)...
  Geocoded 10/135 addresses...
  Geocoded 20/135 addresses...
  Geocoded 30/135 addresses...
  Geocoded 40/135 addresses...
  Geocoded 50/135 addresses...
  Geocoded 60/135 addresses...
  Geocoded 70/135 addresses...
  Geocoded 80/135 addresses...
  Geocoded 90/135 addresses...
  Geocoded 100/135 addresses...
  Geocoded 110/135 addresses...
  Geocoded 120/135 addresses...
  Geocoded 130/135 addresses...
✓ Filled/enriched 135 addresses using Nominatim reverse geocoding
ℹ️  Note: 96 addresses still lack house numbers (expected for public spaces)


## Step 10: Generate Primary Keys

Create unique market_id for each market (FM001, FM002, etc.).

In [27]:
print("\n🆔 Generating Primary Keys")
print("-" * 70)

# Generate unique market_id for each market
markets_with_location['market_id'] = [f"FM{str(i+1).zfill(3)}" for i in range(len(markets_with_location))]
print(f"✓ Generated {len(markets_with_location)} unique market IDs (FM001 - FM{len(markets_with_location):03d})")


🆔 Generating Primary Keys
----------------------------------------------------------------------
✓ Generated 182 unique market IDs (FM001 - FM182)


## Step 11: Export to CSV

In [28]:
print("\n💾 Exporting to CSV")
print("-" * 70)

# ONLY export Berlin markets (exclude outside Berlin)
berlin_markets_only = markets_with_location[markets_with_location['district_id'] != '99999999']
print(f"Filtering to Berlin markets only: {len(berlin_markets_only)} out of {len(markets_with_location)}")

# Select final columns
df_export = berlin_markets_only[FINAL_COLUMNS].copy()

# IMPORTANT: Validate that district_id is never empty (should be guaranteed by spatial join)
empty_district_ids = df_export['district_id'].isna().sum()
if empty_district_ids > 0:
    print(f"⚠️  WARNING: {empty_district_ids} markets have empty district_id!")
else:
    print(f"✓ All {len(df_export)} markets have district_id assigned")

# Fill missing values for all columns EXCEPT district_id and neighborhood_id
# (these should already be populated from spatial join)
for col in df_export.columns:
    if col not in ['district_id', 'neighborhood_id', 'market_id']:
        df_export[col] = df_export[col].fillna('Not available').replace('', 'Not available')

df_export.to_csv(OUTPUT_FILE, index=False, encoding='utf-8', na_rep='Not available')

print(f"✓ Exported to: {OUTPUT_FILE}")
print(f"✓ Total markets: {len(df_export)}")
print(f"✓ Columns: {len(df_export.columns)}")


💾 Exporting to CSV
----------------------------------------------------------------------
Filtering to Berlin markets only: 163 out of 182
✓ All 163 markets have district_id assigned
✓ Exported to: food_markets/berlin_food_markets_clean.csv
✓ Total markets: 163
✓ Columns: 18


## Step 12: Data Quality Validation

In [29]:
print("\n✅ Validating Data Quality")
print("-" * 70)

# Verify no missing values
verify_df = pd.read_csv(OUTPUT_FILE, keep_default_na=False, na_values=[])
missing = verify_df.isnull().sum().sum()
print(f"✓ Missing values: {missing}")

# Verify required columns exist
required = ['market_id', 'neighborhood_id', 'district_id']
for col in required:
    if col in verify_df.columns:
        print(f"✓ {col} column present")
    else:
        print(f"✗ {col} column MISSING!")

# Verify district_id integrity
# IMPORTANT: Convert district_id to string for comparison (pandas reads it as int64)
verify_df['district_id'] = verify_df['district_id'].astype(str)
valid_district_ids = list(DISTRICT_MAPPING.values())  # Only Berlin districts (no 99999999)
invalid_districts = verify_df[~verify_df['district_id'].isin(valid_district_ids)]
if len(invalid_districts) > 0:
    print(f"⚠️  WARNING: {len(invalid_districts)} markets have invalid district_id!")
else:
    print(f"✓ All district_ids are valid (12 Berlin districts)")

# Verify all markets are in Berlin
berlin_count = verify_df['district_id'].isin(valid_district_ids).sum()
print(f"✓ All {berlin_count} markets are in Berlin (no outside markets exported)")

print("\n" + "=" * 70)
print("✨ TRANSFORMATION COMPLETE!")
print("=" * 70)


✅ Validating Data Quality
----------------------------------------------------------------------
✓ Missing values: 0
✓ market_id column present
✓ neighborhood_id column present
✓ district_id column present
✓ All district_ids are valid (12 Berlin districts)
✓ All 163 markets are in Berlin (no outside markets exported)

✨ TRANSFORMATION COMPLETE!


## Final Summary

In [30]:
print(f"\n📊 FINAL SUMMARY:")
print(f"- Total markets: {len(verify_df)}")
print(f"- Market types: {verify_df['market_type'].value_counts().to_dict()}")
print(f"- All markets are in Berlin (outside markets excluded)")
print(f"- Data sources: {verify_df['data_source'].value_counts().to_dict()}")
print(f"- File size: {Path(OUTPUT_FILE).stat().st_size / 1024:.1f} KB")

# Show sample
print(f"\nSample data:")
verify_df.head(3)


📊 FINAL SUMMARY:
- Total markets: 163
- Market types: {'weekly_market': 113, 'christmas_market': 39, 'food_court': 11}
- All markets are in Berlin (outside markets excluded)
- Data sources: {'OSM': 110, 'berlin.de_official': 53}
- File size: 57.0 KB

Sample data:


,market_id,market_name,market_type,address,postal_code,district,neighborhood_id,district_id,opening_days,opening_hours,operator,contact_email,website,accessibility,latitude,longitude,data_source,notes
0,FM001,Winter am THF,christmas_market,Tempelhofer Damm 53,12101,Tempelhof-Schöneberg,703,11007007,See website,2. Adventswochenende: 05. bis 07.12.2025 \n3. ...,Flughafen Tempelhof (Tempelhof Projekt GmbH),kommunikation@thf-berlin.de,https://www.thf-berlin.de/winter-am-thf,ja,52.477,13.386,berlin.de_official,Die Alte Feuerwache ist barrierefrei von außen...
1,FM002,LGBT*Winterdays und Christmas Avenue Nollendor...,christmas_market,Kleiststraße,10787,Tempelhof-Schöneberg,701,11007007,See website,LGBTQIA* Winterdays:,Rutwiess Events Berlin & LGBT*Aktionsgemeinsch...,Info@Rutwiess-events.de,https://www.christmas-avenue.berlin/,nein,52.500,13.353,berlin.de_official,auf dem Markt sind Holzhecksel gestreut. Sollt...
2,FM003,30. Weihnachtsmarkt auf Lehmanns Bauernhof,christmas_market,Lehmanns Bauernhof \nAlt-Marienfelde 35-37,12277,Tempelhof-Schöneberg,705,11007007,See website,Freitag: 14 - 20 Uhr \nSamstag: 12 - 20 Uhr \n...,Lehmanns Bauernhof E&E Trade Event UG (haftung...,karstenemail@gmx.de,https://www.lehmannsbauernhof.de,Ja,52.413,13.366,berlin.de_official,"ebenerdiges Gelände, teilweise Kopfsteinpflast..."


## Next Steps

The dataset is now ready for **Step 3: Database Integration**

Required database schema:
- **Primary Key:** `market_id` (FM001, FM002, etc.)
- **Foreign Keys:** 
  - `district_id` → references `berlin_data.districts(district_id)`
  - `neighborhood_id` → spatial codes (0101, 0102, etc.)
- **Constraints:** NOT NULL on district_id and market_name